# 04 — Comparison Plots for KoVAE Synthetic Methods


```text
500 windows × 2 sec shift ≈ 1000 sec
```

A real subject has many more windows, so its stitched timeline can be much longer.

This updated notebook adds:

```python
COMPARISON_CONFIG["match_duration_to_synthetic"] = True
COMPARISON_CONFIG["crop_duration_sec"] = None
COMPARISON_CONFIG["crop_start_sec"] = 0.0
```

When `match_duration_to_synthetic=True`, the three-way comparison plots are cropped to the common available duration, usually the synthetic subject duration. This makes the visual comparison fairer.

It still supports both KoVAE methods:

```text
rollout_v1
posterior_bank_v2
```


In [1]:

# ============================================================
# 04_comparison_kovae_dual_methods.py
#
# Adapted from friend's comparison.ipynb.
#
# Native-rate comparison for KoVAE synthetic subjects.
#
# Supports:
#   - rollout_v1
#   - posterior_bank_v2
#
# What it compares:
#   1. Real spaced timeline
#   2. Real stitched timeline
#   3. Synthetic timeline
#
# Native rates:
#   ACC:       32 Hz, window 256, shift 64
#   BVP:       64 Hz, window 512, shift 128
#   EDA/TEMP:   4 Hz, window 32,  shift 8
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple

import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Config
# ============================================================

COMPARISON_CONFIG = {
    "project_root": "/home/iailab42/khans1/projects/ir",

    "real_dir": "data/processed/native_rates",
    "synthetic_base_dir": "data/synthetic_subjects/kovae",

    # Run both, or use only one:
    # ["rollout_v1"]
    # ["posterior_bank_v2"]
    # ["rollout_v1", "posterior_bank_v2"]
    "methods_to_compare": ["rollout_v1", "posterior_bank_v2"],

    "figures_base_dir": "figures/comparison",
    "results_base_dir": "results/comparison",

    # Real subject to reconstruct.
    "real_subject_to_plot": "S1",

    # Optional synthetic subject selection.
    # If None, the first available subject from each method is used.
    # Example:
    # {
    #   "rollout_v1": "KOVAE_SYN_rollout_v1_001",
    #   "posterior_bank_v2": "KOVAE_SYN_posterior_bank_v2_001"
    # }
    "synthetic_subject_to_plot_by_method": {
        "rollout_v1": None,
        "posterior_bank_v2": None,
    },

    # Friend's original code used Hann weighting.
    "weight_mode": "hann",

    # For faster plotting on large real subjects, set to 2, 4, or 8.
    "plot_step": 1,

    # Crop real and synthetic comparison plots to the same duration.
    # This fixes the visual issue where real subject timelines can be ~7000 sec
    # while one generated synthetic subject is only ~1000 sec.
    "match_duration_to_synthetic": True,

    # None means: use synthetic duration automatically.
    # Example: 1000 means crop all three plots to first 1000 seconds.
    "crop_duration_sec": None,

    # Usually keep 0.0. Can be changed if you want to start later in the timeline.
    "crop_start_sec": 0.0,

    # User-requested style switch.
    "save_plot": True,

    # Continuous arrays can be large, but useful for later checking.
    "save_continuous_arrays": True,
}


# ============================================================
# Derived paths
# ============================================================

def get_project_paths(config: Dict) -> Dict[str, Path]:
    root = Path(config["project_root"])
    return {
        "root": root,
        "real_dir": root / config["real_dir"],
        "synthetic_base_dir": root / config["synthetic_base_dir"],
        "figures_base_dir": root / config["figures_base_dir"],
        "results_base_dir": root / config["results_base_dir"],
        "configs_dir": root / "configs",
    }


def get_method_paths(paths: Dict[str, Path], method_name: str) -> Dict[str, Path]:
    return {
        "synthetic_dir": paths["synthetic_base_dir"] / method_name,
        "figures_dir": paths["figures_base_dir"] / method_name,
        "results_dir": paths["results_base_dir"] / method_name,
    }


# ============================================================
# Branch configs
# ============================================================

BRANCH_CONFIGS = {
    "ACC": {
        "real_filename": "all_X_acc_32hz.npy",
        "syn_filename": "generated_subjects_X_acc_32hz.npy",
        "hz": 32,
        "window_len": 256,
        "shift_len": 64,
        "channel_names": ["ACC_x", "ACC_y", "ACC_z"],
        "start_col": "acc_start_sample_32hz",
        "end_col": "acc_end_sample_32hz",
    },
    "BVP": {
        "real_filename": "all_X_bvp_64hz.npy",
        "syn_filename": "generated_subjects_X_bvp_64hz.npy",
        "hz": 64,
        "window_len": 512,
        "shift_len": 128,
        "channel_names": ["BVP"],
        "start_col": "bvp_start_sample_64hz",
        "end_col": "bvp_end_sample_64hz",
    },
    "SLOW": {
        "real_filename": "all_X_slow_4hz.npy",
        "syn_filename": "generated_subjects_X_slow_4hz.npy",
        "hz": 4,
        "window_len": 32,
        "shift_len": 8,
        "channel_names": ["EDA", "TEMP"],
        "start_col": "slow_start_sample_4hz",
        "end_col": "slow_end_sample_4hz",
    },
}


# ============================================================
# Safety helpers
# ============================================================

def require_file(path: Path) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return path


def make_dirs(*dirs: Path) -> None:
    for directory in dirs:
        directory.mkdir(parents=True, exist_ok=True)


def check_branch_shape(X: np.ndarray, expected_window_len: int, expected_channels: int, name: str) -> None:
    if X.ndim != 3:
        raise ValueError(f"{name}: expected [N, T, C], got {X.shape}")

    if X.shape[1] != expected_window_len:
        raise ValueError(
            f"{name}: expected window length {expected_window_len}, got {X.shape[1]}"
        )

    if X.shape[2] != expected_channels:
        raise ValueError(
            f"{name}: expected {expected_channels} channels, got {X.shape[2]}"
        )


def print_label_counts(name: str, y: np.ndarray) -> None:
    y = np.asarray(y)
    print(f"{name} label counts:", dict(zip(*np.unique(y, return_counts=True))))


def print_subject_counts(name: str, subjects: np.ndarray) -> None:
    subjects = np.asarray(subjects).astype(str)
    print(f"{name} subject counts:", dict(zip(*np.unique(subjects, return_counts=True))))


def save_json(data: Dict, path: Path) -> None:
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


# ============================================================
# Metadata helpers
# ============================================================

def add_fallback_timing_columns(
    metadata: pd.DataFrame,
    subject_col: str,
    branch_configs: Dict,
    window_in_subject_col: Optional[str] = None,
) -> pd.DataFrame:
    """
    Friend's comparison notebook expects start/end sample columns.

    Real metadata from preprocessing should normally contain these columns.
    Synthetic metadata from our KoVAE generation may not contain them.

    If missing, we create compressed/stitched timing:
        start = window_in_subject * shift_len
        end   = start + window_len
    """

    metadata = metadata.copy()

    if window_in_subject_col is None or window_in_subject_col not in metadata.columns:
        metadata["_fallback_window_order"] = metadata.groupby(subject_col).cumcount()
        order_col = "_fallback_window_order"
    else:
        order_col = window_in_subject_col

    if "start_time_sec" not in metadata.columns:
        metadata["start_time_sec"] = metadata[order_col].astype(float) * 2.0

    if "end_time_sec" not in metadata.columns:
        metadata["end_time_sec"] = metadata["start_time_sec"] + 8.0

    for _, cfg in branch_configs.items():
        start_col = cfg["start_col"]
        end_col = cfg["end_col"]

        if start_col not in metadata.columns:
            metadata[start_col] = metadata[order_col].astype(np.int64) * int(cfg["shift_len"])

        if end_col not in metadata.columns:
            metadata[end_col] = metadata[start_col].astype(np.int64) + int(cfg["window_len"])

    return metadata


def standardize_real_metadata(
    metadata: pd.DataFrame,
    y: np.ndarray,
    subjects: np.ndarray,
) -> pd.DataFrame:
    metadata = metadata.copy()

    if "subject" not in metadata.columns:
        metadata["subject"] = subjects.astype(str)
    else:
        metadata["subject"] = subjects.astype(str)

    if "activity_label" not in metadata.columns:
        metadata["activity_label"] = y.astype(np.int64)
    else:
        metadata["activity_label"] = y.astype(np.int64)

    metadata["array_index"] = np.arange(len(y), dtype=np.int64)

    metadata = add_fallback_timing_columns(
        metadata=metadata,
        subject_col="subject",
        branch_configs=BRANCH_CONFIGS,
        window_in_subject_col=None,
    )

    return metadata


def standardize_synthetic_metadata(
    metadata: pd.DataFrame,
    y: np.ndarray,
    subjects: np.ndarray,
) -> pd.DataFrame:
    metadata = metadata.copy()

    if "synthetic_subject" not in metadata.columns:
        metadata["synthetic_subject"] = subjects.astype(str)
    else:
        metadata["synthetic_subject"] = subjects.astype(str)

    if "activity_label" not in metadata.columns:
        metadata["activity_label"] = y.astype(np.int64)
    else:
        metadata["activity_label"] = y.astype(np.int64)

    metadata["array_index"] = np.arange(len(y), dtype=np.int64)

    window_col = None

    if "synthetic_window_in_subject" in metadata.columns:
        window_col = "synthetic_window_in_subject"
    elif "window_in_subject" in metadata.columns:
        window_col = "window_in_subject"

    metadata = add_fallback_timing_columns(
        metadata=metadata,
        subject_col="synthetic_subject",
        branch_configs=BRANCH_CONFIGS,
        window_in_subject_col=window_col,
    )

    return metadata


# ============================================================
# Loading helpers
# ============================================================

def load_common_real(real_dir: Path) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    y_path = real_dir / "all_y.npy"
    subject_path = real_dir / "all_subject.npy"
    metadata_path = real_dir / "all_metadata.csv"

    require_file(y_path)
    require_file(subject_path)

    y = np.load(y_path).astype(np.int64)
    subjects = np.load(subject_path, allow_pickle=True).astype(str)

    if metadata_path.exists():
        metadata = pd.read_csv(metadata_path)
    else:
        warnings.warn(
            f"Real metadata not found at {metadata_path}. "
            "Creating fallback stitched metadata from subject order."
        )
        metadata = pd.DataFrame()

    if len(y) != len(subjects):
        raise ValueError(f"Real y/subject length mismatch: {len(y)} vs {len(subjects)}")

    if len(metadata) not in {0, len(y)}:
        raise ValueError(f"Real metadata/y length mismatch: {len(metadata)} vs {len(y)}")

    if len(metadata) == 0:
        metadata = pd.DataFrame(index=np.arange(len(y)))

    metadata = standardize_real_metadata(metadata, y, subjects)

    return y, subjects, metadata


def load_common_synthetic(synthetic_dir: Path) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    y_path = synthetic_dir / "generated_subjects_all_y.npy"
    subject_path = synthetic_dir / "generated_subjects_all_subject.npy"
    metadata_path = synthetic_dir / "generated_subjects_metadata.csv"

    require_file(y_path)
    require_file(subject_path)
    require_file(metadata_path)

    y = np.load(y_path).astype(np.int64)
    subjects = np.load(subject_path, allow_pickle=True).astype(str)
    metadata = pd.read_csv(metadata_path)

    if len(y) != len(subjects):
        raise ValueError(f"Synthetic y/subject length mismatch: {len(y)} vs {len(subjects)}")

    if len(metadata) != len(y):
        raise ValueError(f"Synthetic metadata/y length mismatch: {len(metadata)} vs {len(y)}")

    metadata = standardize_synthetic_metadata(metadata, y, subjects)

    return y, subjects, metadata


def load_branch_arrays(real_dir: Path, synthetic_dir: Path) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    real_X = {}
    syn_X = {}

    for branch_name, cfg in BRANCH_CONFIGS.items():
        real_path = real_dir / cfg["real_filename"]
        syn_path = synthetic_dir / cfg["syn_filename"]

        require_file(real_path)
        require_file(syn_path)

        Xr = np.load(real_path).astype(np.float32)
        Xs = np.load(syn_path).astype(np.float32)

        expected_channels = len(cfg["channel_names"])

        check_branch_shape(
            X=Xr,
            expected_window_len=cfg["window_len"],
            expected_channels=expected_channels,
            name=f"Real {branch_name}",
        )

        check_branch_shape(
            X=Xs,
            expected_window_len=cfg["window_len"],
            expected_channels=expected_channels,
            name=f"Synthetic {branch_name}",
        )

        real_X[branch_name] = Xr
        syn_X[branch_name] = Xs

    return real_X, syn_X


def load_all_data_for_method(
    real_dir: Path,
    synthetic_dir: Path,
) -> Tuple[
    Dict[str, np.ndarray],
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
    Dict[str, np.ndarray],
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
]:
    print("=" * 70)
    print("Loading real native-rate data")

    real_y, real_subjects, real_meta = load_common_real(real_dir)

    print("Real y:", real_y.shape)
    print("Real subjects:", real_subjects.shape)
    print("Real metadata:", real_meta.shape)
    print_label_counts("Real", real_y)
    print_subject_counts("Real", real_subjects)

    print("\n" + "=" * 70)
    print("Loading synthetic native-rate data")
    print("Synthetic dir:", synthetic_dir)

    syn_y, syn_subjects, syn_meta = load_common_synthetic(synthetic_dir)

    print("Synthetic y:", syn_y.shape)
    print("Synthetic subjects:", syn_subjects.shape)
    print("Synthetic metadata:", syn_meta.shape)
    print_label_counts("Synthetic", syn_y)
    print_subject_counts("Synthetic", syn_subjects)

    print("\n" + "=" * 70)
    print("Loading branch arrays")

    real_X, syn_X = load_branch_arrays(real_dir, synthetic_dir)

    for branch_name in BRANCH_CONFIGS:
        print(f"Real {branch_name}:", real_X[branch_name].shape)
        print(f"Synthetic {branch_name}:", syn_X[branch_name].shape)

        if len(real_X[branch_name]) != len(real_y):
            raise ValueError(
                f"Real {branch_name}/y length mismatch: "
                f"{len(real_X[branch_name])} vs {len(real_y)}"
            )

        if len(syn_X[branch_name]) != len(syn_y):
            raise ValueError(
                f"Synthetic {branch_name}/y length mismatch: "
                f"{len(syn_X[branch_name])} vs {len(syn_y)}"
            )

    return real_X, real_y, real_subjects, real_meta, syn_X, syn_y, syn_subjects, syn_meta


# ============================================================
# Overlap-add reconstruction
# ============================================================

def make_window_weights(window_len: int, weight_mode: str = "hann") -> np.ndarray:
    if weight_mode == "hann":
        weights = np.hanning(window_len).astype(np.float32)
        weights = np.maximum(weights, 1e-3)

    elif weight_mode == "ones":
        weights = np.ones(window_len, dtype=np.float32)

    else:
        raise ValueError("weight_mode must be 'hann' or 'ones'.")

    return weights[:, None]


def reconstruct_overlap_add(
    windows: np.ndarray,
    labels: np.ndarray,
    start_samples: np.ndarray,
    end_samples: np.ndarray,
    window_len: int,
    weight_mode: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    windows = np.asarray(windows, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int64)
    start_samples = np.asarray(start_samples, dtype=np.int64)
    end_samples = np.asarray(end_samples, dtype=np.int64)

    if windows.ndim != 3:
        raise ValueError(f"Expected windows [N,T,C], got {windows.shape}")

    if windows.shape[1] != window_len:
        raise ValueError(f"Expected window_len={window_len}, got {windows.shape[1]}")

    if len(windows) != len(labels):
        raise ValueError("windows/labels length mismatch.")

    if len(windows) != len(start_samples):
        raise ValueError("windows/start_samples length mismatch.")

    if len(windows) != len(end_samples):
        raise ValueError("windows/end_samples length mismatch.")

    if len(windows) == 0:
        raise ValueError("No windows given for reconstruction.")

    offset = int(start_samples.min())
    start_samples = start_samples - offset
    end_samples = end_samples - offset

    total_len = int(end_samples.max())
    num_channels = windows.shape[2]

    continuous_sum = np.zeros((total_len, num_channels), dtype=np.float32)
    weight_sum = np.zeros((total_len, 1), dtype=np.float32)

    max_label = max(8, int(labels.max()))
    label_scores = np.zeros((total_len, max_label + 1), dtype=np.float32)

    win_weight = make_window_weights(window_len, weight_mode=weight_mode)

    for i in range(len(windows)):
        s = int(start_samples[i])
        e = int(end_samples[i])

        if e <= s:
            continue

        if e - s != window_len:
            raise ValueError(f"Window {i} has length {e - s}, expected {window_len}.")

        continuous_sum[s:e] += windows[i] * win_weight
        weight_sum[s:e] += win_weight

        lab = int(labels[i])

        if 0 <= lab < label_scores.shape[1]:
            label_scores[s:e, lab] += win_weight[:, 0]

    covered = weight_sum[:, 0] > 1e-8

    continuous = np.full_like(continuous_sum, np.nan, dtype=np.float32)
    continuous[covered] = continuous_sum[covered] / weight_sum[covered]

    sample_labels = np.zeros(total_len, dtype=np.int64)
    sample_labels[covered] = np.argmax(label_scores[covered], axis=1)

    coverage = weight_sum[:, 0]

    return continuous, sample_labels, coverage


# ============================================================
# Subject extraction
# ============================================================

def get_real_subject_windows_spaced(
    X: np.ndarray,
    metadata: pd.DataFrame,
    subject_name: str,
    cfg: Dict,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    sub = metadata[metadata["subject"] == subject_name].copy()

    if len(sub) == 0:
        available = sorted(metadata["subject"].unique().tolist())
        raise ValueError(f"Real subject {subject_name} not found. Available: {available}")

    start_col = cfg["start_col"]
    end_col = cfg["end_col"]

    sub = sub.sort_values(start_col).reset_index(drop=True)

    idxs = sub["array_index"].to_numpy(dtype=np.int64)
    windows = X[idxs]
    labels = sub["activity_label"].to_numpy(dtype=np.int64)

    start_samples = sub[start_col].to_numpy(dtype=np.int64)
    end_samples = sub[end_col].to_numpy(dtype=np.int64)

    return windows, labels, start_samples, end_samples, sub


def get_real_subject_windows_stitched(
    X: np.ndarray,
    metadata: pd.DataFrame,
    subject_name: str,
    cfg: Dict,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    sub = metadata[metadata["subject"] == subject_name].copy()

    if len(sub) == 0:
        available = sorted(metadata["subject"].unique().tolist())
        raise ValueError(f"Real subject {subject_name} not found. Available: {available}")

    start_col = cfg["start_col"]

    sub = sub.sort_values(start_col).reset_index(drop=True)

    idxs = sub["array_index"].to_numpy(dtype=np.int64)
    windows = X[idxs]
    labels = sub["activity_label"].to_numpy(dtype=np.int64)

    start_samples = np.arange(len(sub), dtype=np.int64) * int(cfg["shift_len"])
    end_samples = start_samples + int(cfg["window_len"])

    return windows, labels, start_samples, end_samples, sub


def get_synthetic_subject_windows(
    X: np.ndarray,
    metadata: pd.DataFrame,
    subject_name: str,
    cfg: Dict,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    sub = metadata[metadata["synthetic_subject"] == subject_name].copy()

    if len(sub) == 0:
        available = sorted(metadata["synthetic_subject"].unique().tolist())
        raise ValueError(
            f"Synthetic subject {subject_name} not found. Available: {available[:20]}"
        )

    if "synthetic_window_in_subject" in sub.columns:
        sub = sub.sort_values("synthetic_window_in_subject").reset_index(drop=True)
    elif "window_in_subject" in sub.columns:
        sub = sub.sort_values("window_in_subject").reset_index(drop=True)
    else:
        sub = sub.sort_values(cfg["start_col"]).reset_index(drop=True)

    idxs = sub["array_index"].to_numpy(dtype=np.int64)
    windows = X[idxs]
    labels = sub["activity_label"].to_numpy(dtype=np.int64)

    start_samples = sub[cfg["start_col"]].to_numpy(dtype=np.int64)
    end_samples = sub[cfg["end_col"]].to_numpy(dtype=np.int64)

    return windows, labels, start_samples, end_samples, sub


# ============================================================
# Plotting helpers
# ============================================================

def activity_segments(sample_labels: np.ndarray) -> List[Tuple[int, int, int]]:
    labels = np.asarray(sample_labels, dtype=np.int64)

    if len(labels) == 0:
        return []

    segments = []
    start = 0
    current = int(labels[0])

    for i in range(1, len(labels)):
        lab = int(labels[i])

        if lab != current:
            segments.append((start, i, current))
            start = i
            current = lab

    segments.append((start, len(labels), current))

    return segments


def add_activity_background(ax, sample_labels: np.ndarray, target_hz: float) -> None:
    ymin, ymax = ax.get_ylim()

    if not np.isfinite(ymin) or not np.isfinite(ymax) or ymin == ymax:
        return

    y_text = ymin + 0.03 * (ymax - ymin)

    for start, end, lab in activity_segments(sample_labels):
        if lab == 0:
            continue

        start_sec = start / float(target_hz)
        end_sec = end / float(target_hz)

        ax.axvspan(start_sec, end_sec, alpha=0.10)

        if end_sec - start_sec >= 10:
            ax.text(
                (start_sec + end_sec) / 2.0,
                y_text,
                f"act {lab}",
                ha="center",
                va="bottom",
                fontsize=8,
            )


def decimate_for_plot(
    time_sec: np.ndarray,
    signal: np.ndarray,
    labels: np.ndarray,
    step: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if step <= 1:
        return time_sec, signal, labels

    return time_sec[::step], signal[::step], labels[::step]


def crop_continuous_by_seconds(
    continuous: np.ndarray,
    labels: np.ndarray,
    target_hz: int,
    crop_start_sec: float,
    crop_duration_sec: Optional[float],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Crop one continuous reconstructed signal and its sample labels.

    If crop_duration_sec is None, no crop is applied.
    """

    if crop_duration_sec is None:
        return continuous, labels

    start_idx = int(round(float(crop_start_sec) * float(target_hz)))
    end_idx = start_idx + int(round(float(crop_duration_sec) * float(target_hz)))

    start_idx = max(0, start_idx)
    end_idx = min(len(continuous), end_idx)

    if end_idx <= start_idx:
        raise ValueError(
            f"Invalid crop range: start_idx={start_idx}, end_idx={end_idx}, "
            f"len={len(continuous)}, target_hz={target_hz}"
        )

    return continuous[start_idx:end_idx], labels[start_idx:end_idx]


def compute_common_crop_duration_sec(
    real_spaced_cont: np.ndarray,
    real_stitched_cont: np.ndarray,
    syn_cont: np.ndarray,
    target_hz: int,
    config: Dict,
) -> Optional[float]:
    """
    Decide how many seconds to plot.

    If match_duration_to_synthetic=True and crop_duration_sec=None,
    this uses the available synthetic duration. It also makes sure we do not
    exceed the real timelines.
    """

    if not bool(config.get("match_duration_to_synthetic", False)):
        return None

    requested_duration = config.get("crop_duration_sec", None)

    real_spaced_sec = len(real_spaced_cont) / float(target_hz)
    real_stitched_sec = len(real_stitched_cont) / float(target_hz)
    syn_sec = len(syn_cont) / float(target_hz)

    max_available = min(real_spaced_sec, real_stitched_sec, syn_sec)

    if requested_duration is None:
        return float(max_available)

    return float(min(float(requested_duration), max_available))


def finalize_plot(fig: plt.Figure, save_path: Path, save_plot: bool) -> None:
    if save_plot:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print("Saved:", save_path)
    else:
        plt.show()


def plot_one_continuous_branch(
    continuous: np.ndarray,
    sample_labels: np.ndarray,
    title: str,
    save_path: Path,
    channel_names: List[str],
    target_hz: int,
    plot_step: int,
    save_plot: bool,
    crop_duration_sec: Optional[float] = None,
    crop_start_sec: float = 0.0,
) -> None:
    continuous, sample_labels = crop_continuous_by_seconds(
        continuous=continuous,
        labels=sample_labels,
        target_hz=target_hz,
        crop_start_sec=crop_start_sec,
        crop_duration_sec=crop_duration_sec,
    )

    time_sec = np.arange(len(continuous)) / float(target_hz)

    if crop_duration_sec is not None:
        title = f"{title} | cropped to {len(continuous) / float(target_hz):.1f} sec"

    time_plot, cont_plot, labels_plot = decimate_for_plot(
        time_sec,
        continuous,
        sample_labels,
        plot_step,
    )

    fig, axes = plt.subplots(
        len(channel_names),
        1,
        figsize=(18, max(4, 3 * len(channel_names))),
        sharex=True,
    )

    if len(channel_names) == 1:
        axes = [axes]

    for c, name in enumerate(channel_names):
        ax = axes[c]

        ax.plot(time_plot, cont_plot[:, c], linewidth=0.7)
        ax.set_ylabel(name)
        ax.set_title(f"{title} | {name}")

        add_activity_background(
            ax,
            labels_plot,
            target_hz=target_hz / max(plot_step, 1),
        )

    axes[-1].set_xlabel("Time in seconds")

    plt.tight_layout()
    finalize_plot(fig, save_path, save_plot=save_plot)


def plot_three_way_channel(
    real_spaced_cont: np.ndarray,
    real_spaced_labels: np.ndarray,
    real_stitched_cont: np.ndarray,
    real_stitched_labels: np.ndarray,
    syn_cont: np.ndarray,
    syn_labels: np.ndarray,
    channel_idx: int,
    channel_name: str,
    save_path: Path,
    target_hz: int,
    plot_step: int,
    save_plot: bool,
    method_name: str,
    crop_duration_sec: Optional[float] = None,
    crop_start_sec: float = 0.0,
) -> None:
    real_spaced_cont, real_spaced_labels = crop_continuous_by_seconds(
        continuous=real_spaced_cont,
        labels=real_spaced_labels,
        target_hz=target_hz,
        crop_start_sec=crop_start_sec,
        crop_duration_sec=crop_duration_sec,
    )
    real_stitched_cont, real_stitched_labels = crop_continuous_by_seconds(
        continuous=real_stitched_cont,
        labels=real_stitched_labels,
        target_hz=target_hz,
        crop_start_sec=crop_start_sec,
        crop_duration_sec=crop_duration_sec,
    )
    syn_cont, syn_labels = crop_continuous_by_seconds(
        continuous=syn_cont,
        labels=syn_labels,
        target_hz=target_hz,
        crop_start_sec=crop_start_sec,
        crop_duration_sec=crop_duration_sec,
    )

    real_spaced_time = np.arange(len(real_spaced_cont)) / float(target_hz)
    real_stitched_time = np.arange(len(real_stitched_cont)) / float(target_hz)
    syn_time = np.arange(len(syn_cont)) / float(target_hz)

    real_spaced_time_plot, real_spaced_cont_plot, real_spaced_labels_plot = decimate_for_plot(
        real_spaced_time,
        real_spaced_cont,
        real_spaced_labels,
        plot_step,
    )

    real_stitched_time_plot, real_stitched_cont_plot, real_stitched_labels_plot = decimate_for_plot(
        real_stitched_time,
        real_stitched_cont,
        real_stitched_labels,
        plot_step,
    )

    syn_time_plot, syn_cont_plot, syn_labels_plot = decimate_for_plot(
        syn_time,
        syn_cont,
        syn_labels,
        plot_step,
    )

    fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=False)

    axes[0].plot(
        real_spaced_time_plot,
        real_spaced_cont_plot[:, channel_idx],
        linewidth=0.7,
    )
    axes[0].set_title(f"Real spaced timeline | {channel_name}")
    axes[0].set_ylabel(channel_name)
    add_activity_background(
        axes[0],
        real_spaced_labels_plot,
        target_hz=target_hz / max(plot_step, 1),
    )

    axes[1].plot(
        real_stitched_time_plot,
        real_stitched_cont_plot[:, channel_idx],
        linewidth=0.7,
    )
    axes[1].set_title(f"Real stitched timeline, no activity-0 gaps | {channel_name}")
    axes[1].set_ylabel(channel_name)
    add_activity_background(
        axes[1],
        real_stitched_labels_plot,
        target_hz=target_hz / max(plot_step, 1),
    )

    axes[2].plot(
        syn_time_plot,
        syn_cont_plot[:, channel_idx],
        linewidth=0.7,
    )
    axes[2].set_title(f"Synthetic subject ({method_name}) | {channel_name}")
    axes[2].set_ylabel(channel_name)
    axes[2].set_xlabel("Time in seconds")
    add_activity_background(
        axes[2],
        syn_labels_plot,
        target_hz=target_hz / max(plot_step, 1),
    )

    plt.tight_layout()
    finalize_plot(fig, save_path, save_plot=save_plot)


def save_continuous_arrays(
    prefix: str,
    branch_name: str,
    continuous: np.ndarray,
    sample_labels: np.ndarray,
    coverage: np.ndarray,
    results_dir: Path,
    save_arrays: bool,
) -> None:
    if not save_arrays:
        return

    branch_name_clean = branch_name.lower()

    x_path = results_dir / f"{prefix}_{branch_name_clean}_continuous.npy"
    y_path = results_dir / f"{prefix}_{branch_name_clean}_continuous_activity_labels.npy"
    cov_path = results_dir / f"{prefix}_{branch_name_clean}_continuous_coverage.npy"

    np.save(x_path, continuous)
    np.save(y_path, sample_labels)
    np.save(cov_path, coverage)

    print("Saved:", x_path)
    print("Saved:", y_path)
    print("Saved:", cov_path)


def print_reconstruction_summary(
    name: str,
    windows: np.ndarray,
    continuous: np.ndarray,
    labels: np.ndarray,
    target_hz: int,
) -> Dict:
    duration_sec = len(continuous) / float(target_hz)

    summary = {
        "windows_shape": list(windows.shape),
        "continuous_shape": list(continuous.shape),
        "duration_sec": float(duration_sec),
        "duration_min": float(duration_sec / 60),
        "duration_hours": float(duration_sec / 3600),
        "label_counts": {int(k): int(v) for k, v in zip(*np.unique(labels, return_counts=True))},
    }

    print(f"{name} windows:", windows.shape)
    print(f"{name} continuous:", continuous.shape)
    print(
        f"{name} duration: {duration_sec:.2f} sec = "
        f"{duration_sec / 60:.2f} min = {duration_sec / 3600:.2f} h"
    )
    print(f"{name} label counts:", summary["label_counts"])

    return summary


# ============================================================
# One branch pipeline
# ============================================================

def process_branch(
    branch_name: str,
    cfg: Dict,
    real_X_branch: np.ndarray,
    real_meta: pd.DataFrame,
    syn_X_branch: np.ndarray,
    syn_meta: pd.DataFrame,
    real_subject_to_plot: str,
    syn_subject_to_plot: str,
    method_name: str,
    figures_dir: Path,
    results_dir: Path,
    config: Dict,
) -> Dict:
    print("\n" + "=" * 70)
    print(f"Processing branch: {branch_name}")
    print(f"Method: {method_name}")
    print(f"Hz: {cfg['hz']}")
    print(f"Window length: {cfg['window_len']}")
    print(f"Shift length: {cfg['shift_len']}")
    print(f"Channels: {cfg['channel_names']}")

    hz = int(cfg["hz"])
    window_len = int(cfg["window_len"])
    channel_names = cfg["channel_names"]

    weight_mode = config["weight_mode"]
    plot_step = int(config["plot_step"])
    save_plot = bool(config["save_plot"])
    save_arrays = bool(config["save_continuous_arrays"])

    # ------------------------------------------------------------
    # Real spaced timeline
    # ------------------------------------------------------------

    print("\n" + "-" * 70)
    print(f"Reconstructing REAL SPACED subject: {real_subject_to_plot}")
    print("Mode: preserve original native-rate start/end samples if available.")

    (
        real_spaced_windows,
        real_spaced_labels,
        real_spaced_start,
        real_spaced_end,
        _,
    ) = get_real_subject_windows_spaced(
        X=real_X_branch,
        metadata=real_meta,
        subject_name=real_subject_to_plot,
        cfg=cfg,
    )

    (
        real_spaced_cont,
        real_spaced_sample_labels,
        real_spaced_coverage,
    ) = reconstruct_overlap_add(
        windows=real_spaced_windows,
        labels=real_spaced_labels,
        start_samples=real_spaced_start,
        end_samples=real_spaced_end,
        window_len=window_len,
        weight_mode=weight_mode,
    )

    real_spaced_summary = print_reconstruction_summary(
        name=f"Real spaced {branch_name}",
        windows=real_spaced_windows,
        continuous=real_spaced_cont,
        labels=real_spaced_labels,
        target_hz=hz,
    )

    save_continuous_arrays(
        prefix=f"real_{real_subject_to_plot}_spaced",
        branch_name=branch_name,
        continuous=real_spaced_cont,
        sample_labels=real_spaced_sample_labels,
        coverage=real_spaced_coverage,
        results_dir=results_dir,
        save_arrays=save_arrays,
    )

    plot_one_continuous_branch(
        continuous=real_spaced_cont,
        sample_labels=real_spaced_sample_labels,
        title=f"Real {real_subject_to_plot} spaced timeline | {branch_name}",
        save_path=figures_dir / f"real_{real_subject_to_plot}_spaced_{branch_name}_all_channels.png",
        channel_names=channel_names,
        target_hz=hz,
        plot_step=plot_step,
        save_plot=save_plot,
    )

    # ------------------------------------------------------------
    # Real stitched/compressed timeline
    # ------------------------------------------------------------

    print("\n" + "-" * 70)
    print(f"Reconstructing REAL STITCHED subject: {real_subject_to_plot}")
    print("Mode: compressed kept windows using native-rate shift.")

    (
        real_stitched_windows,
        real_stitched_labels,
        real_stitched_start,
        real_stitched_end,
        _,
    ) = get_real_subject_windows_stitched(
        X=real_X_branch,
        metadata=real_meta,
        subject_name=real_subject_to_plot,
        cfg=cfg,
    )

    (
        real_stitched_cont,
        real_stitched_sample_labels,
        real_stitched_coverage,
    ) = reconstruct_overlap_add(
        windows=real_stitched_windows,
        labels=real_stitched_labels,
        start_samples=real_stitched_start,
        end_samples=real_stitched_end,
        window_len=window_len,
        weight_mode=weight_mode,
    )

    real_stitched_summary = print_reconstruction_summary(
        name=f"Real stitched {branch_name}",
        windows=real_stitched_windows,
        continuous=real_stitched_cont,
        labels=real_stitched_labels,
        target_hz=hz,
    )

    save_continuous_arrays(
        prefix=f"real_{real_subject_to_plot}_stitched",
        branch_name=branch_name,
        continuous=real_stitched_cont,
        sample_labels=real_stitched_sample_labels,
        coverage=real_stitched_coverage,
        results_dir=results_dir,
        save_arrays=save_arrays,
    )

    plot_one_continuous_branch(
        continuous=real_stitched_cont,
        sample_labels=real_stitched_sample_labels,
        title=f"Real {real_subject_to_plot} stitched timeline | {branch_name}",
        save_path=figures_dir / f"real_{real_subject_to_plot}_stitched_{branch_name}_all_channels.png",
        channel_names=channel_names,
        target_hz=hz,
        plot_step=plot_step,
        save_plot=save_plot,
    )

    # ------------------------------------------------------------
    # Synthetic timeline
    # ------------------------------------------------------------

    print("\n" + "-" * 70)
    print(f"Reconstructing SYNTHETIC subject: {syn_subject_to_plot}")
    print(f"Method: {method_name}")

    (
        syn_windows,
        syn_labels,
        syn_start,
        syn_end,
        _,
    ) = get_synthetic_subject_windows(
        X=syn_X_branch,
        metadata=syn_meta,
        subject_name=syn_subject_to_plot,
        cfg=cfg,
    )

    syn_cont, syn_sample_labels, syn_coverage = reconstruct_overlap_add(
        windows=syn_windows,
        labels=syn_labels,
        start_samples=syn_start,
        end_samples=syn_end,
        window_len=window_len,
        weight_mode=weight_mode,
    )

    syn_summary = print_reconstruction_summary(
        name=f"Synthetic {method_name} {branch_name}",
        windows=syn_windows,
        continuous=syn_cont,
        labels=syn_labels,
        target_hz=hz,
    )

    common_crop_duration_sec = compute_common_crop_duration_sec(
        real_spaced_cont=real_spaced_cont,
        real_stitched_cont=real_stitched_cont,
        syn_cont=syn_cont,
        target_hz=hz,
        config=config,
    )
    crop_start_sec = float(config.get("crop_start_sec", 0.0))

    if common_crop_duration_sec is not None:
        print(
            f"Cropping comparison plots to {common_crop_duration_sec:.2f} seconds "
            f"from start={crop_start_sec:.2f} sec."
        )

    save_continuous_arrays(
        prefix=f"{method_name}_{syn_subject_to_plot}",
        branch_name=branch_name,
        continuous=syn_cont,
        sample_labels=syn_sample_labels,
        coverage=syn_coverage,
        results_dir=results_dir,
        save_arrays=save_arrays,
    )

    plot_one_continuous_branch(
        continuous=syn_cont,
        sample_labels=syn_sample_labels,
        title=f"Synthetic {method_name} {syn_subject_to_plot} | {branch_name}",
        save_path=figures_dir / f"{method_name}_{syn_subject_to_plot}_{branch_name}_all_channels.png",
        channel_names=channel_names,
        target_hz=hz,
        plot_step=plot_step,
        save_plot=save_plot,
        crop_duration_sec=common_crop_duration_sec,
        crop_start_sec=crop_start_sec,
    )

    # ------------------------------------------------------------
    # Three-way channel comparison plots
    # ------------------------------------------------------------

    print("\n" + "-" * 70)
    print(f"Saving three-way comparison plots for {branch_name}")

    for channel_idx, channel_name in enumerate(channel_names):
        plot_three_way_channel(
            real_spaced_cont=real_spaced_cont,
            real_spaced_labels=real_spaced_sample_labels,
            real_stitched_cont=real_stitched_cont,
            real_stitched_labels=real_stitched_sample_labels,
            syn_cont=syn_cont,
            syn_labels=syn_sample_labels,
            channel_idx=channel_idx,
            channel_name=channel_name,
            save_path=figures_dir / (
                f"real_spaced_vs_stitched_vs_"
                f"{method_name}_{syn_subject_to_plot}_{branch_name}_{channel_name}.png"
            ),
            target_hz=hz,
            plot_step=plot_step,
            save_plot=save_plot,
            method_name=method_name,
            crop_duration_sec=common_crop_duration_sec,
            crop_start_sec=crop_start_sec,
        )

    return {
        "real_spaced": real_spaced_summary,
        "real_stitched": real_stitched_summary,
        "synthetic": syn_summary,
        "comparison_crop_duration_sec": common_crop_duration_sec,
        "comparison_crop_start_sec": crop_start_sec,
    }


# ============================================================
# One method pipeline
# ============================================================

def choose_synthetic_subject(
    method_name: str,
    syn_subjects: np.ndarray,
    config: Dict,
) -> str:
    selected = config["synthetic_subject_to_plot_by_method"].get(method_name)

    available = sorted(np.unique(syn_subjects.astype(str)).tolist())

    if selected is not None:
        if selected not in set(available):
            raise ValueError(
                f"Selected synthetic subject {selected} not found for {method_name}. "
                f"Available: {available[:20]}"
            )
        return selected

    if len(available) == 0:
        raise ValueError(f"No synthetic subjects found for {method_name}")

    return available[0]


def process_one_method(method_name: str, config: Dict) -> Dict:
    paths = get_project_paths(config)
    method_paths = get_method_paths(paths, method_name)

    make_dirs(method_paths["figures_dir"], method_paths["results_dir"], paths["configs_dir"])

    print("\n" + "#" * 90)
    print(f"COMPARISON METHOD: {method_name}")
    print("#" * 90)
    print("Real dir:", paths["real_dir"])
    print("Synthetic dir:", method_paths["synthetic_dir"])
    print("Figures dir:", method_paths["figures_dir"])
    print("Results dir:", method_paths["results_dir"])

    (
        real_X,
        real_y,
        real_subjects,
        real_meta,
        syn_X,
        syn_y,
        syn_subjects,
        syn_meta,
    ) = load_all_data_for_method(
        real_dir=paths["real_dir"],
        synthetic_dir=method_paths["synthetic_dir"],
    )

    real_subject_to_plot = config["real_subject_to_plot"]

    print("\nAvailable real subjects:")
    print(sorted(np.unique(real_subjects).tolist()))

    print("\nAvailable synthetic subjects:")
    print(sorted(np.unique(syn_subjects).tolist())[:30])

    if real_subject_to_plot not in set(real_subjects.tolist()):
        raise ValueError(
            f"real_subject_to_plot={real_subject_to_plot} not found. "
            f"Available: {sorted(np.unique(real_subjects).tolist())}"
        )

    syn_subject_to_plot = choose_synthetic_subject(
        method_name=method_name,
        syn_subjects=syn_subjects,
        config=config,
    )

    print("\nSelected real subject:", real_subject_to_plot)
    print("Selected synthetic subject:", syn_subject_to_plot)

    branch_summaries = {}

    for branch_name, cfg in BRANCH_CONFIGS.items():
        branch_summaries[branch_name] = process_branch(
            branch_name=branch_name,
            cfg=cfg,
            real_X_branch=real_X[branch_name],
            real_meta=real_meta,
            syn_X_branch=syn_X[branch_name],
            syn_meta=syn_meta,
            real_subject_to_plot=real_subject_to_plot,
            syn_subject_to_plot=syn_subject_to_plot,
            method_name=method_name,
            figures_dir=method_paths["figures_dir"],
            results_dir=method_paths["results_dir"],
            config=config,
        )

    method_summary = {
        "method_name": method_name,
        "real_subject_to_plot": real_subject_to_plot,
        "synthetic_subject_to_plot": syn_subject_to_plot,
        "real_dir": str(paths["real_dir"]),
        "synthetic_dir": str(method_paths["synthetic_dir"]),
        "figures_dir": str(method_paths["figures_dir"]),
        "results_dir": str(method_paths["results_dir"]),
        "branch_summaries": branch_summaries,
    }

    save_json(method_summary, method_paths["results_dir"] / "comparison_summary.json")

    print("\n" + "=" * 70)
    print(f"Done for method: {method_name}")
    print("Plots saved in:", method_paths["figures_dir"])
    print("Results saved in:", method_paths["results_dir"])

    return method_summary


# ============================================================
# Main
# ============================================================

def main(config: Dict = COMPARISON_CONFIG) -> Dict:
    paths = get_project_paths(config)
    make_dirs(paths["figures_base_dir"], paths["results_base_dir"], paths["configs_dir"])

    save_json(config, paths["configs_dir"] / "comparison_kovae_dual_methods_config.json")

    all_summaries = {}

    for method_name in config["methods_to_compare"]:
        all_summaries[method_name] = process_one_method(method_name, config)

    combined_summary = {
        "methods_compared": config["methods_to_compare"],
        "summaries": all_summaries,
    }

    save_json(combined_summary, paths["results_base_dir"] / "combined_comparison_summary.json")

    print("\n" + "#" * 90)
    print("All comparison methods completed.")
    print("Combined summary:", paths["results_base_dir"] / "combined_comparison_summary.json")
    print("#" * 90)

    return combined_summary


if __name__ == "__main__":
    outputs = main(COMPARISON_CONFIG)



##########################################################################################
COMPARISON METHOD: rollout_v1
##########################################################################################
Real dir: /home/iailab42/khans1/projects/ir/data/processed/native_rates
Synthetic dir: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae/rollout_v1
Figures dir: /home/iailab42/khans1/projects/ir/figures/comparison/rollout_v1
Results dir: /home/iailab42/khans1/projects/ir/results/comparison/rollout_v1
Loading real native-rate data
Real y: (46925,)
Real subjects: (46925,)
Real metadata: (46925, 19)
Real label counts: {np.int64(1): np.int64(4538), np.int64(2): np.int64(3206), np.int64(3): np.int64(2279), np.int64(4): np.int64(3442), np.int64(5): np.int64(6809), np.int64(6): np.int64(13520), np.int64(7): np.int64(4663), np.int64(8): np.int64(8468)}
Real subject counts: {np.str_('S1'): np.int64(3445), np.str_('S10'): np.int64(3532), np.str_('S11'): np.int64(3490), np.

## Run comparison with cropping

To compare both methods and crop real timelines to the synthetic duration:

```python
COMPARISON_CONFIG["methods_to_compare"] = ["rollout_v1", "posterior_bank_v2"]
COMPARISON_CONFIG["match_duration_to_synthetic"] = True
COMPARISON_CONFIG["crop_duration_sec"] = None
COMPARISON_CONFIG["crop_start_sec"] = 0.0
COMPARISON_CONFIG["save_plot"] = True
```

To manually crop all timelines to 1000 seconds:

```python
COMPARISON_CONFIG["crop_duration_sec"] = 1000
```

To disable cropping:

```python
COMPARISON_CONFIG["match_duration_to_synthetic"] = False
```


In [2]:
COMPARISON_CONFIG["methods_to_compare"] = ["rollout_v1", "posterior_bank_v2"]

# Crop real and synthetic plots to the same duration.
COMPARISON_CONFIG["match_duration_to_synthetic"] = True
COMPARISON_CONFIG["crop_duration_sec"] = None
COMPARISON_CONFIG["crop_start_sec"] = 0.0

COMPARISON_CONFIG["save_plot"] = True

outputs = main(COMPARISON_CONFIG)
outputs



##########################################################################################
COMPARISON METHOD: rollout_v1
##########################################################################################
Real dir: /home/iailab42/khans1/projects/ir/data/processed/native_rates
Synthetic dir: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae/rollout_v1
Figures dir: /home/iailab42/khans1/projects/ir/figures/comparison/rollout_v1
Results dir: /home/iailab42/khans1/projects/ir/results/comparison/rollout_v1
Loading real native-rate data
Real y: (46925,)
Real subjects: (46925,)
Real metadata: (46925, 19)
Real label counts: {np.int64(1): np.int64(4538), np.int64(2): np.int64(3206), np.int64(3): np.int64(2279), np.int64(4): np.int64(3442), np.int64(5): np.int64(6809), np.int64(6): np.int64(13520), np.int64(7): np.int64(4663), np.int64(8): np.int64(8468)}
Real subject counts: {np.str_('S1'): np.int64(3445), np.str_('S10'): np.int64(3532), np.str_('S11'): np.int64(3490), np.

{'methods_compared': ['rollout_v1', 'posterior_bank_v2'],
 'summaries': {'rollout_v1': {'method_name': 'rollout_v1',
   'real_subject_to_plot': 'S1',
   'synthetic_subject_to_plot': 'KOVAE_SYN_rollout_v1_001',
   'real_dir': '/home/iailab42/khans1/projects/ir/data/processed/native_rates',
   'synthetic_dir': '/home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae/rollout_v1',
   'figures_dir': '/home/iailab42/khans1/projects/ir/figures/comparison/rollout_v1',
   'results_dir': '/home/iailab42/khans1/projects/ir/results/comparison/rollout_v1',
   'branch_summaries': {'ACC': {'real_spaced': {'windows_shape': [3445,
       256,
       3],
      'continuous_shape': [284480, 3],
      'duration_sec': 8890.0,
      'duration_min': 148.16666666666666,
      'duration_hours': 2.4694444444444446,
      'label_counts': {1: 347,
       2: 141,
       3: 170,
       4: 203,
       5: 442,
       6: 1175,
       7: 375,
       8: 592}},
     'real_stitched': {'windows_shape': [3445, 256, 3